# PrimeKV — Quick Test Notebook

Run these cells top-to-bottom to install, test, and interactively
compare KV cache strategies. Works on CPU (free tier) or GPU.

**From your phone:** just tap each cell and hit the play button.

## 1. Clone and install

In [ ]:
!rm -rf /content/PrimeKV
!git clone https://github.com/arunvenkatadri/PrimeKV.git
%cd /content/PrimeKV
!git checkout claude/scaffold-primekv-Q9QKN
!pip install -e ".[dev,web]" -q

## 2. Run unit tests (no network, no GPU, ~3 seconds)

In [ ]:
!pytest tests/ -v

## 3. Run the comparison CLI with real GPT-2

Downloads GPT-2 (124M) on first run (~500 MB). Takes 30-60s on CPU.

In [ ]:
!python benchmarks/compare.py --model gpt2 --decode-tokens 16 --max-length 64

## 4. Run comparison from Python (more control)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from primekv.eval import Workload, run_comparison
from primekv.cache import PrimeKVCache
from primekv.classifier import RuleBasedClassifier, Tier
from primekv.baselines import FullCache, H2OCache, UniformQuantCache

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

num_layers = model.config.n_layer

caches = {
    "full":        FullCache(num_layers),
    "uniform_int4": UniformQuantCache(num_layers, bits=4),
    "h2o":         H2OCache(num_layers, capacity=32),
    "primekv":     PrimeKVCache(
        num_layers=num_layers,
        classifier=RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3),
        max_entries_per_tier={Tier.SUPPORTING: 32},
    ),
}

workload = Workload(
    prompt="System: you are a helpful assistant. User: What is the capital of France? Assistant:",
    decode_tokens=16,
    max_length=64,
)

report = run_comparison(caches, workload, model, tok)
print(report.to_markdown())
print()
for r in report.results:
    if r.generated:
        print(f"--- {r.name} ---")
        print(r.generated)
        print()

## 5. Launch interactive Gradio UI

This creates a **public share link** you can open in any browser tab
(or send to a collaborator). The link is active as long as this cell
is running.

In [ ]:
from webui.app import build_demo

demo = build_demo()
demo.launch(share=True)

## 6. Inspect PrimeKV tier distribution

In [ ]:
from primekv.metrics import tier_distribution, summarize_stats

# Use the primekv cache from step 4 (still in memory)
pkv = caches["primekv"]

print("Tier distribution:")
for tier, count in tier_distribution(pkv).items():
    print(f"  {tier:12s}  {count} tokens")

print("\nCache stats:")
for k, v in summarize_stats(pkv.stats).items():
    print(f"  {str(k):20s}  {v}")

## 7. Real-scale validation (GPU required)

This section runs the 2D eviction × quantization sweep on a **real instruction-tuned model** (Phi-2, 2.7B) with a long prompt (~1500 tokens). This is what goes into the paper.

**Before running this section:** go to `Runtime → Change runtime type → T4 GPU` (or A100 if you have Colab Pro). Then restart the runtime and run cell 1 again to reinstall.

Expected runtime: ~5-15 minutes on T4, ~2-5 minutes on A100.

In [ ]:
import torch

# Verify GPU is available.
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, "
    "then restart the runtime and re-run cells 1 and 7.1."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 7.2 Load a real model

Phi-2 (2.7B) is instruction-tuned, open-weight, and doesn't require HuggingFace authentication. It's a genuine step up from GPT-2 and fits on a free T4.

If you want to use Llama-3 or Mistral instead, you'll need to:
1. Create an HF token at https://huggingface.co/settings/tokens
2. Add it as a Colab secret named `HF_TOKEN`
3. Replace `"microsoft/phi-2"` below with `"meta-llama/Llama-3.2-3B-Instruct"` or similar

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "microsoft/phi-2"  # 2.7B, instruction-tuned, no auth required

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # fp16 to fit in T4 VRAM
    device_map="cuda",
    trust_remote_code=True,
    attn_implementation="eager", # Important: we need standard past_key_values
)
model.eval()
print(f"Loaded {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Heads: {model.config.num_attention_heads}")
print(f"Max context: {model.config.max_position_embeddings}")

### 7.3 A real long-context prompt

This is ~1500 tokens of Wikipedia-style content about the Eiffel Tower. It has the structure we want:
- Named entities early (Anchor tier should save them)
- Lots of factual elaboration (Supporting tier material)
- A continuation the model should complete fluently

In [ ]:
LONG_PROMPT = """The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889. Locally nicknamed La dame de fer, it was constructed as the centerpiece of the 1889 World's Fair and to crown the centennial anniversary of the French Revolution. Although initially criticized by some of France's leading artists and intellectuals for its design, it has since become a global cultural icon of France and one of the most recognizable structures in the world. The Eiffel Tower is the most-visited paid monument in the world; 6.91 million people ascended it in 2015.

The tower is 330 meters tall, about the same height as an 81-story building, and the tallest structure in Paris. Its base is square, measuring 125 meters on each side. During its construction, the Eiffel Tower surpassed the Washington Monument to become the tallest man-made structure in the world, a title it held for 41 years until the Chrysler Building in New York City was finished in 1930. It was the first structure to reach a height of 300 meters. Due to the addition of a broadcasting aerial at the top of the tower in 1957, it is now taller than the Chrysler Building by 5.2 meters.

The tower has three levels for visitors, with restaurants on the first and second levels. The top level's upper platform is 276 meters above the ground, the highest observation deck accessible to the public in the European Union. Tickets can be purchased to ascend by stairs or elevator to the first and second levels. The climb from ground level to the first level is over 300 steps, as is the climb from the first level to the second. Although there is a staircase to the top level, it is usually accessible only by elevator.

Eiffel openly acknowledged that inspiration for the tower came from the Latting Observatory built in New York City in 1853. In May 1884, working at home, Maurice Koechlin, a senior engineer at the Compagnie des Etablissements Eiffel, made a sketch of their idea, described by him as a great pylon, consisting of four lattice girders standing apart at the base and coming together at the top, joined together by metal trusses at regular intervals.

Gustave Eiffel initially showed little enthusiasm for the project, but he did approve further study, and the two engineers then asked Stephen Sauvestre, the head of the company's architectural department, to contribute to the design. Sauvestre added decorative arches to the base of the tower, a glass pavilion to the first level, and other embellishments. Eiffel bought the rights to the patent on 13 September 1884. By 30 March 1885, Eiffel presented his plans to the Société des Ingénieurs Civils; after discussing the technical problems and emphasising the practical uses of the tower, he finished his talk by saying the tower would symbolise not only the art of the modern engineer, but also the century of industry and science in which we are living.

The proposed tower had been a subject of controversy, drawing criticism from those who did not believe it was feasible and those who objected on artistic grounds. These objections were an expression of a long-standing debate in France about the relationship between architecture and engineering. It came to a head as work began at the Champ de Mars: a Committee of Three Hundred led by the prominent architect Charles Garnier and including some of the most important figures of the arts, such as Adolphe Bouguereau, Guy de Maupassant, Charles Gounod and Jules Massenet, sent a petition to Jean-Charles Alphand, the Minister of Works and Commissioner for the Exhibition, and it was published by Le Temps on 14 February 1887.

Some of the protests had such remarkable foresight that they would prove nearly prophetic. Gustave Eiffel himself later wrote that the tower would endure because it embodies"""

# Show the token count
tokens = tok(LONG_PROMPT, return_tensors="pt")
print(f"Prompt token count: {tokens.input_ids.shape[-1]}")

### 7.4 Run the 2D eviction × quantization sweep

This is the headline experiment. It runs ~27 configurations (1 full + 2 uniform + 4 h2o + 4 streamingllm + 4×3 primekv) against the same long prompt and produces the Pareto scatter plot.

**Expected runtime: 5-15 min on T4, 2-5 min on A100.**

In [ ]:
from primekv.sweep import sweep_2d_tradeoff, plot_report
import logging
logging.basicConfig(level=logging.WARNING)

report = sweep_2d_tradeoff(
    model=model,
    tokenizer=tok,
    prompt=LONG_PROMPT,
    eviction_caps=[64, 128, 256, 512],    # aggressive to moderate compression
    precisions=["fp16", "int8", "int4"],
    decode_tokens=16,
    max_length=1024,                       # truncate to 1024 tokens
    device="cuda",
    progress=lambda msg: print(f"  {msg}"),
)

print(f"\nDone. {len(report.points)} configurations.")

### 7.5 Plot the results

In [ ]:
import matplotlib.pyplot as plt

fig = plot_report(report, output_path="primekv_2d_sweep.png")
plt.show()

# Also save the raw numbers
with open("primekv_2d_sweep.csv", "w") as f:
    f.write(report.to_csv())
print("\nSaved: primekv_2d_sweep.png, primekv_2d_sweep.csv")

### 7.6 Headline numbers table

In [ ]:
import pandas as pd

# Build a clean dataframe for the LinkedIn post / paper
rows = []
for p in report.points:
    rows.append({
        "cache": p.cache,
        "cap": p.extra.get("cap"),
        "precision": p.extra.get("precision"),
        "memory_MB": round(p.memory_bytes / (1024 * 1024), 2),
        "ratio": round(p.compression_ratio, 2),
        "ppl": round(p.perplexity, 3) if p.perplexity else None,
        "prefill_ms": round(p.prefill_ms, 1),
        "decode_ms": round(p.decode_ms, 1),
    })
df = pd.DataFrame(rows)
df = df.sort_values(["cache", "cap", "precision"])
print(df.to_string(index=False))